# Lab Tasks - Solutions

In this task, we will work with the same dataset from Lab 4, which contains information about contacts between patients and healthcare workers in a hospital ward in France. The data was gathered using wearable sensors that could detect face-to-face close-range proximity of participants wearing the sensors.

The raw data is provided as two tab-separated files:

1. `hospital-metadata.csv`: One line per participant in the hospital ward, where each participant has a unique numeric ID and a "status" (e.g., patient, doctor, etc.).
2. `hospital-contacts.csv`: Each line indicates a contact event between two participants, with the time of the event and the numeric IDs of the participants involved.

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

### Task 1

Load the hospital metadata and contact data into two separate Pandas DataFrames.

In [ ]:
# read the metadata
df_metadata = pd.read_csv("hospital-metadata.csv", sep="\t").set_index("participant_id")
df_metadata.head(5)

In [ ]:
# read the contact events
df_contacts = pd.read_csv("hospital-contacts.csv", sep="\t")
df_contacts.head(5)

### Task 2

Create a weighted undirected network from the two DataFrames, such that:

- There is a node for every participant in the study. Each node should have an attribute indicating their "status".
- Edges between nodes have a "weight", indicating the number of times two participants (nodes) have been in contact.

What size is the resulting network?

In [ ]:
# create the undirected network and add the nodes from the metadata
g = nx.Graph()
for participant_id, row in df_metadata.iterrows():
    g.add_node(participant_id, status=row["status"])

In [ ]:
# count frequency of contact between each pair of participants
from collections import Counter
counts = Counter()
for i, row in df_contacts.iterrows():
    node1 = row["participant1"]
    node2 = row["participant2"]
    # represent the pair as a frozen set, so it's unique and hashable
    pair = frozenset([node1, node2])
    counts[pair] += 1

In [ ]:
# add weighted edges for each pair
for p in counts:
    pair = list(p)
    node1, node2 = pair[0], pair[1]
    g.add_edge(node1, node2, weight=counts[p])

In [ ]:
# check size of the network
print(f"Network: {g.number_of_nodes()} nodes and {g.number_of_edges()} edges")

### Task 3

Using the network created in Task 2, create two different **subgraphs**:

1. The subgraph containing only doctors.
2. The subgraph containing only doctors and patients.

What size are the two resulting subgraphs?

In [ ]:
# find the nodes representing doctors
doctor_nodes = [n for n, attr in g.nodes(data=True) if attr["status"] == "doctor"]
# create a subgraph with the filtered nodes
g_doctors = g.subgraph(doctor_nodes)
# check size of the subgraph
print(f"Doctors subgraph: {g_doctors.number_of_nodes()} nodes and {g_doctors.number_of_edges()} edges")

In [ ]:
# find the nodes representing doctors and patients
doctor_patient_nodes = [n for n, attr in g.nodes(data=True) if attr["status"] in ["doctor", "patient"]]
# create a subgraph with the filtered nodes
g_doctors_patients = g.subgraph(doctor_patient_nodes)
# check size of the subgraph
print(f"Doctors + Patients subgraph: {g_doctors_patients.number_of_nodes()} nodes and {g_doctors_patients.number_of_edges()} edges")

### Task 4

Identify the student with the highest weighted degree score in the overall network. Construct the **ego network** for this node.

Use NetworkX to produce a visualisation of this ego network in your notebook.

In [ ]:
# get list of student nodes
student_nodes = [n for n, attr in g.nodes(data=True) if attr["status"] == "student"]
# calculate the weighted degrees
wdegrees = g.degree(student_nodes, weight="weight")
# convert it to a DataFrame for inspection
wdegree_list = list(wdegrees)
df_wdegree = pd.DataFrame(wdegree_list, columns=["Node", "Degree"]).set_index("Node")
# sort the DataFrame by highest weighted degree
df_wdegree.sort_values(by="Degree", ascending=False, inplace=True)
df_wdegree

In [ ]:
# create the ego network for the student with the highest weight
student_node_id = df_wdegree.index[0]
print(f"Creating ego network for node {student_node_id} ...")
eg = nx.ego_graph(g, student_node_id)
print(f"Ego network: {eg.number_of_nodes()} nodes and {eg.number_of_edges()} edges")

In [ ]:
# visualise the ego network
plt.figure(figsize=(9, 7))
plt.margins(0.1, 0.1)
# position all nodes
pos = nx.spring_layout(eg)
# draw the full network
nx.draw_networkx(eg, pos, with_labels=False, node_size=100, node_color="lightblue")
# draw the ego in red, with larger node size
nx.draw_networkx_nodes(eg, pos, nodelist=[student_node_id], node_size=500, node_color="coral")
plt.axis("off")
plt.show()

### Task 5

Export the ego network from Task 4 as a new GEXF file.

Load the GEXF file in **Gephi** and experiment with different layout algorithms to produce a final visualisation of the ego network.

In [ ]:
# save the ego network as a GEXF file
nx.write_gexf(eg, "ego.gexf")